# Bluebikes Stations

For a given `YYYY-MM` month (defaults to the latest complete calendar month),
looks at station-level activity three ways:

1. **All stations**: a Folium map of every station that started a trip in the
   month, sized by average daily ridership.
2. **New stations**: stations that appear as a `start_station_name` that month
   but not in the month before, i.e. stations that opened that month.
3. **Top stations, weekday vs weekend**: stations ranked by average daily
   ridership separately for weekdays and weekends, with the top 10 of each
   written to a Markdown report (optionally posted to Slack) and mapped.

Pulls monthly trip-history data from the [Bluebikes/Hubway System Data bucket](https://s3.amazonaws.com/hubway-data/index.html). Files are named `{YYYYMM}-bluebikes-tripdata.zip` and are available from `201805` onward.

In [ ]:
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import pandas as pd
import requests

from plots import add_legend, add_station_marker, base_map
from shared.plots.style import PALETTE

## Params

In [ ]:
MONTH = None  # YYYY-MM, or None for the latest complete calendar month
SEND_SLACK = False  # post the top-stations report to Slack
SLACK_CHANNEL = "#data-science"

## Load or fetch trip data

Fetches the target month plus the month before it (for the new-station diff).
Each month is cached independently at
`data/raw/bluebikes_tripdata/{yyyymm}/data.csv`, so re-running the notebook
only fetches months not already on disk.

In [ ]:
BASE_URL = "https://s3.amazonaws.com/hubway-data"


def fetch_month_tripdata(yyyymm):
    raw_path = Path(f"data/raw/bluebikes_tripdata/{yyyymm}/data.csv")

    if not raw_path.exists():
        # Almost every month is "{yyyymm}-bluebikes-tripdata.zip"; a rare
        # month (e.g. 202511) ships as "{yyyymm}-bluebikes-tripdata.csv.zip" instead.
        for suffix in ("-bluebikes-tripdata.zip", "-bluebikes-tripdata.csv.zip"):
            url = f"{BASE_URL}/{yyyymm}{suffix}"
            response = requests.get(url, timeout=60)
            if response.ok:
                break
        response.raise_for_status()

        with ZipFile(BytesIO(response.content)) as zf:
            csv_name = next(
                name
                for name in zf.namelist()
                if name.endswith(".csv") and "__MACOSX" not in name
            )
            raw_path.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(csv_name) as src, open(raw_path, "wb") as dst:
                dst.write(src.read())

        print(f"Fetched and cached: {raw_path}")
    else:
        print(f"Loaded cached pull: {raw_path}")

    return raw_path


def latest_complete_month():
    first_of_this_month = pd.Timestamp.today().normalize().replace(day=1)
    return (first_of_this_month - pd.DateOffset(months=1)).strftime("%Y-%m")


month = MONTH or latest_complete_month()
month_label = pd.Period(month, freq="M").strftime("%B %Y")
prior_month = (pd.Period(month, freq="M") - 1).strftime("%Y-%m")

month_path = fetch_month_tripdata(pd.Period(month, freq="M").strftime("%Y%m"))
prior_month_path = fetch_month_tripdata(pd.Period(prior_month, freq="M").strftime("%Y%m"))

trips = pd.read_csv(
    month_path,
    usecols=["started_at", "start_station_name", "start_lat", "start_lng"],
)
trips["trip_date"] = trips["started_at"].str.slice(0, 10)

prior_station_names = pd.read_csv(prior_month_path, usecols=["start_station_name"])[
    "start_station_name"
].unique()

output_dir = Path(f"outputs/{month}")
output_dir.mkdir(parents=True, exist_ok=True)

trips.head()

## Per-station stats, whole month

"Days used" is the count of distinct calendar days with at least one trip
starting at the station; "daily ridership" is total trips at the station
divided by days used (its average ridership on a day it saw any activity);
rank is by daily ridership among all stations that started a trip in the
month.

In [ ]:
station_stats = (
    trips.groupby("start_station_name")
    .agg(
        total_trips=("trip_date", "size"),
        days_used=("trip_date", "nunique"),
        lat=("start_lat", "median"),
        lng=("start_lng", "median"),
    )
    .reset_index()
    .rename(columns={"start_station_name": "station"})
)
station_stats["daily_ridership"] = station_stats["total_trips"] / station_stats["days_used"]
station_stats["rank"] = station_stats["daily_ridership"].rank(ascending=False, method="min").astype(int)
station_stats["is_new"] = ~station_stats["station"].isin(prior_station_names)
station_stats = station_stats.sort_values("rank")

total_stations = len(station_stats)
print(f"{total_stations:,} stations started a trip in {month}")
station_stats.head()

## Map: all stations

Every station active in the month, with marker area proportional to daily
ridership. Stations new this month are highlighted.

In [ ]:
ALL_STATION_COLORS = {"Existing station": PALETTE[0], "New this month": PALETTE[1]}

# Scale by sqrt so marker *area* tracks ridership; clamp to a readable range.
max_ridership = station_stats["daily_ridership"].max()
radii = 2 + 10 * np.sqrt(station_stats["daily_ridership"] / max_ridership)

all_stations_map = base_map(station_stats["lat"].tolist(), station_stats["lng"].tolist(), zoom_start=12)
for row, radius in zip(station_stats.itertuples(), radii):
    popup_html = (
        f"<b>{row.station}</b><br>"
        f"Daily ridership: {row.daily_ridership:.1f} trips/day<br>"
        f"Total trips: {row.total_trips:,}<br>"
        f"Days used in {month}: {row.days_used}<br>"
        f"Rank: {row.rank:,} of {total_stations:,} stations"
    )
    color = ALL_STATION_COLORS["New this month" if row.is_new else "Existing station"]
    add_station_marker(all_stations_map, row.lat, row.lng, color, popup_html, row.station, radius=radius)

add_legend(all_stations_map, ALL_STATION_COLORS)

map_path = output_dir / "all_stations_map.html"
all_stations_map.save(map_path)
print(f"Saved map: {map_path}")
display(all_stations_map)

## New stations

Stations that started a trip in the month but not in the month before.

In [ ]:
new_stations = station_stats[station_stats["is_new"]].copy()
print(f"{len(new_stations)} new station(s) in {month} vs {prior_month}")
new_stations

In [ ]:
if new_stations.empty:
    print(f"No new stations in {month} vs {prior_month}; skipping map.")
else:
    new_stations_map = base_map(new_stations["lat"].tolist(), new_stations["lng"].tolist())
    for row in new_stations.itertuples():
        popup_html = (
            f"<b>{row.station}</b><br>"
            f"Days used in {month}: {row.days_used}<br>"
            f"Daily ridership: {row.daily_ridership:.1f} trips/day<br>"
            f"Total trips: {row.total_trips:,}<br>"
            f"Rank: {row.rank:,} of {total_stations:,} stations"
        )
        add_station_marker(new_stations_map, row.lat, row.lng, PALETTE[0], popup_html, row.station)

    map_path = output_dir / "new_stations_map.html"
    new_stations_map.save(map_path)
    print(f"Saved map: {map_path}")
    display(new_stations_map)

## Top stations, weekday vs weekend

Same stats as above, but computed separately per day type: "days used" counts
only days of that type, and rank is among all stations active on that day
type.

In [ ]:
trips["day_type"] = np.where(
    pd.to_datetime(trips["trip_date"]).dt.dayofweek >= 5, "weekend", "weekday"
)

station_day_stats = (
    trips.groupby(["start_station_name", "day_type"])
    .agg(
        total_trips=("trip_date", "size"),
        days_used=("trip_date", "nunique"),
        lat=("start_lat", "median"),
        lng=("start_lng", "median"),
    )
    .reset_index()
)
station_day_stats["daily_ridership"] = (
    station_day_stats["total_trips"] / station_day_stats["days_used"]
)
station_day_stats["rank"] = (
    station_day_stats.groupby("day_type")["daily_ridership"]
    .rank(ascending=False, method="min")
    .astype(int)
)

station_counts = station_day_stats.groupby("day_type")["start_station_name"].nunique()
weekday_station_count = int(station_counts.get("weekday", 0))
weekend_station_count = int(station_counts.get("weekend", 0))


def day_type_table(day_type):
    return (
        station_day_stats[station_day_stats["day_type"] == day_type]
        .set_index("start_station_name")[
            ["total_trips", "days_used", "daily_ridership", "rank", "lat", "lng"]
        ]
        .add_prefix(f"{day_type}_")
    )


day_type_stats = day_type_table("weekday").join(day_type_table("weekend"), how="outer")
day_type_stats["lat"] = day_type_stats[["weekday_lat", "weekend_lat"]].mean(axis=1)
day_type_stats["lng"] = day_type_stats[["weekday_lng", "weekend_lng"]].mean(axis=1)
day_type_stats = day_type_stats.drop(
    columns=["weekday_lat", "weekday_lng", "weekend_lat", "weekend_lng"]
)
day_type_stats = day_type_stats.reset_index().rename(
    columns={"start_station_name": "station"}
)

top_weekday = (
    day_type_stats.dropna(subset=["weekday_rank"]).sort_values("weekday_rank").head(10)
)
top_weekend = (
    day_type_stats.dropna(subset=["weekend_rank"]).sort_values("weekend_rank").head(10)
)

overlap_stations = set(top_weekday["station"]) & set(top_weekend["station"])
print(f"{len(overlap_stations)} station(s) rank top 10 on both weekdays and weekends")
top_weekday[["station", "weekday_rank", "weekday_daily_ridership", "weekday_total_trips"]]

### Markdown report

In [ ]:
def format_top_table(df, day_type):
    lines = [
        "| Rank | Station | Daily ridership | Total trips |",
        "|---|---|---|---|",
    ]
    for row in df.itertuples():
        rank = int(getattr(row, f"{day_type}_rank"))
        ridership = getattr(row, f"{day_type}_daily_ridership")
        total_trips = int(getattr(row, f"{day_type}_total_trips"))
        lines.append(
            f"| {rank} | {row.station} | {ridership:.1f} | {total_trips:,} |"
        )
    return "\n".join(lines)


report_lines = [
    f"# Bluebikes Top Stations, {month_label}",
    "",
    (
        f"Top 10 stations by average daily ridership for {month_label}, ranked "
        "separately for weekdays and weekends."
    ),
    "",
    f"## Top 10 weekday stations (of {weekday_station_count:,} active)",
    "",
    format_top_table(top_weekday, "weekday"),
    "",
    f"## Top 10 weekend stations (of {weekend_station_count:,} active)",
    "",
    format_top_table(top_weekend, "weekend"),
    "",
]

if overlap_stations:
    report_lines += [
        "## Overlap",
        "",
        f"{len(overlap_stations)} station(s) rank in both top 10 lists: "
        + ", ".join(sorted(overlap_stations)),
        "",
    ]

report_path = output_dir / "top_stations_report.md"
report_path.write_text("\n".join(report_lines))
print(f"Saved report: {report_path}")
print("\n".join(report_lines))

### Send report to Slack

Only runs when `SEND_SLACK = True` in the params cell.

In [ ]:
if SEND_SLACK:
    from shared.slack import markdown_to_mrkdwn, send_message

    send_message(SLACK_CHANNEL, markdown_to_mrkdwn(report_path.read_text()))
    print(f"Sent report to {SLACK_CHANNEL}")
else:
    print("SEND_SLACK is False; skipping Slack post.")

### Map: top stations, weekday vs weekend

In [ ]:
CATEGORY_COLORS = {
    "weekday": PALETTE[0],
    "weekend": PALETTE[1],
    "overlap": "#D4AF37",  # gold
}
CATEGORY_LABELS = {
    "weekday": "Top 10 weekday only",
    "weekend": "Top 10 weekend only",
    "overlap": "Top 10 both",
}


def categorize(station):
    in_weekday = station in set(top_weekday["station"])
    in_weekend = station in set(top_weekend["station"])
    if in_weekday and in_weekend:
        return "overlap"
    return "weekday" if in_weekday else "weekend"


def fmt_side(row, day_type, total_stations):
    rank = getattr(row, f"{day_type}_rank")
    if pd.isna(rank):
        return f"{day_type.capitalize()}: no trips recorded"
    ridership = getattr(row, f"{day_type}_daily_ridership")
    total_trips = int(getattr(row, f"{day_type}_total_trips"))
    return (
        f"{day_type.capitalize()}: {ridership:.1f} trips/day "
        f"({total_trips:,} total), rank {int(rank):,} of {total_stations:,}"
    )


map_stations = sorted(set(top_weekday["station"]) | set(top_weekend["station"]))
map_df = day_type_stats[day_type_stats["station"].isin(map_stations)].copy()
map_df["category"] = map_df["station"].map(categorize)

top_stations_map = base_map(map_df["lat"].tolist(), map_df["lng"].tolist())
for row in map_df.itertuples():
    popup_html = (
        f"<b>{row.station}</b><br>"
        f"{fmt_side(row, 'weekday', weekday_station_count)}<br>"
        f"{fmt_side(row, 'weekend', weekend_station_count)}"
    )
    add_station_marker(top_stations_map, row.lat, row.lng, CATEGORY_COLORS[row.category], popup_html, row.station)

add_legend(top_stations_map, {CATEGORY_LABELS[cat]: color for cat, color in CATEGORY_COLORS.items()})

map_path = output_dir / "top_stations_map.html"
top_stations_map.save(map_path)
print(f"Saved map: {map_path}")
display(top_stations_map)